In [5]:
!pip install rouge-score
!pip install nltk
!pip install gspread gspread-dataframe
!pip install pyrouge



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: C:\Users\Tanner\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: C:\Users\Tanner\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: C:\Users\Tanner\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: C:\Users\Tanner\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:

from rouge_score import rouge_scorer
import pandas as pd
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
import gspread
from gspread_dataframe import set_with_dataframe
import os
import pyrouge
from pyrouge import Rouge155

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Tanner\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Tanner\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
# os.environ = "/content/drive/My Drive/RA Notebooks/pyrouge/tools/ROUGE-1.5.5/"
# r = pyrouge

In [3]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL', 'rougeLsum'], use_stemmer=True)
score = scorer.score('The quick brown fox jumps over the lazy dog',
                     'The quick brown dog jumps on the log.')

print(score)

#rouge = pyrouge.Rouge155()

{'rouge1': Score(precision=0.75, recall=0.6666666666666666, fmeasure=0.7058823529411765), 'rouge2': Score(precision=0.2857142857142857, recall=0.25, fmeasure=0.26666666666666666), 'rougeL': Score(precision=0.625, recall=0.5555555555555556, fmeasure=0.5882352941176471), 'rougeLsum': Score(precision=0.625, recall=0.5555555555555556, fmeasure=0.5882352941176471)}


In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
df = pd.read_excel("/content/drive/MyDrive/AI PlugIn Research/Reference Information/Product Data/Product Reviews.xlsx")
item1  = "9387779262"
item2  = "B0BJMV9BXJ"
item3  = "B0CFX3X5KD"
item4  = "B0D79QXMN4"
item5  = "B07DKZWC87"
item6  = "B07ND3WR64"
item7  = "B08TR5Z9XY"
item8  = "B09V5NPHP3"
item9  = "B09WZBPX7K"
item10 = "B09ZXJDSL5"

dataSetList = [item2, item3, item4, item5, item6, item7, item8, item9, item10]

In [ ]:
import os
import pandas as pd
from rouge_score import rouge_scorer

# -----------------------------
# Set subreddit here
# -----------------------------
subreddit = "worldnews"  # <-- change this as needed

# -----------------------------
# Paths based on subreddit
# -----------------------------
data_dir = "C:/Users/Tanner/Desktop/summarizer/corpus_algorithms_and_models/corpus_data_downloads/data"
summary_base_dir = "C:/Users/Tanner/Desktop/summarizer/corpus_algorithms_and_models/corpus_summaries/tfidf_conversation_summaries"
rouge_scores_dir = f"C:/Users/Tanner/Desktop/summarizer/corpus_algorithms_and_models/rouge_scores/{subreddit}_scores"

original_csv = os.path.join(data_dir, subreddit, f"{subreddit}_conversations.csv")
tfidf_dir = os.path.join(summary_base_dir, subreddit)

tfidf_scores_csv = os.path.join(rouge_scores_dir, f"{subreddit}_rouge_scores_tfidf_vs_original.csv")

# -----------------------------
# Load dataset
# -----------------------------
df = pd.read_csv(original_csv)
df = df[df["text"].notnull()]
conversation_ids = df["conversation_id"].dropna().unique().tolist()

# -----------------------------
# ROUGE scorer
# -----------------------------
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# -----------------------------
# Compute scores
# -----------------------------
tfidf_scores = []

for convo_id in conversation_ids:
    convo_df = df[df["conversation_id"] == convo_id]
    if convo_df.empty:
        continue

    original_text = "\n".join(str(t).strip() for t in convo_df["text"] if isinstance(t, str))


    # TF-IDF summary
    # Load TF-IDF summary
    tfidf_path = os.path.join(tfidf_dir, f"tfidf_{convo_id}.txt")
    elapsed_seconds = None
    tfidf_summary = ""

    if os.path.exists(tfidf_path):
        with open(tfidf_path, "r", encoding="utf-8") as f:
            lines = f.readlines()

        # Check if the last line contains elapsed time
        if lines[-1].startswith("# Summary elapsed time (seconds):"):
            elapsed_seconds = float(lines[-1].split(":")[1].strip())
            tfidf_summary = "".join(lines[:-1]).strip()  # all except the last line
        else:
            tfidf_summary = "".join(lines).strip()
            elapsed_seconds = None

        # Compute ROUGE
        scores = scorer.score(original_text, tfidf_summary)
        tfidf_scores.append({
            "conversation_id": convo_id,
            "rouge1_precision": scores["rouge1"].precision,
            "rouge1_recall": scores["rouge1"].recall,
            "rouge1_f1": scores["rouge1"].fmeasure,
            "rouge2_precision": scores["rouge2"].precision,
            "rouge2_recall": scores["rouge2"].recall,
            "rouge2_f1": scores["rouge2"].fmeasure,
            "rougeL_precision": scores["rougeL"].precision,
            "rougeL_recall": scores["rougeL"].recall,
            "rougeL_f1": scores["rougeL"].fmeasure,
            "summary_time_seconds": elapsed_seconds
        })


# -----------------------------
# Save CSVs
# -----------------------------
pd.DataFrame(tfidf_scores).to_csv(tfidf_scores_csv, index=False)

print(f"Saved ROUGE scores: TF-IDF → {tfidf_scores_csv}")


Saved ROUGE scores: TF-IDF → C:/Users/Tanner/Desktop/summarizer/corpus_algorithms_and_models/rouge_scores/worldnews_scores\worldnews_rouge_scores_tfidf_vs_original.csv


In [ ]:
def LoadSentences(revs):
    revCounter = 0
    combinedRevs = ''
    senList = []
    for i in revs.index:

      rev = revs.at[i, "content"] #parses out review info
      tempList = nltk.tokenize.sent_tokenize(rev) #splits review into sentences
      for sen in tempList:
        senList.append(sen)

    # print("senList: ")
    # print(senList)
    # senList.sort(key=len)
    # print("sorted: ")
    # print(senList)
    for sen in senList:
      combinedRevs += ' ' + (sen) #adds sentence to combinedRevs

    return combinedRevs

In [ ]:
i = 1
data = []
precisionData = []
recallData = []
fmeasureData = []

for id in dataSetList:
  i += 1
  revs = df.loc[df["asin"] == id]
  #print(revs)

  revPar = LoadSentences(revs);
  basis, tfidf, diversity1, diversity2 = loadSummaries(i, id)
  print("ID: ", id)
  print("Reviews: ", revPar)
  print("Basis Sum: ", basis)
  print("TFIDF Sum: ", tfidf)
  print("Diversity1 Sum: ", diversity1)
  print("Diversity2 Sum: ", diversity2)


  basisScore = scorer.score(basis, revPar)
  tfidfScore = scorer.score(tfidf, revPar)
  diversity1Score = scorer.score(diversity1, revPar)
  diversity2Score = scorer.score(diversity2, revPar)


  # data.append([i, id, 'Basis', 'Rouge 1', basisScore['rouge1'].precision, basisScore['rouge1'].recall, basisScore['rouge1'].fmeasure])
  # data.append([i, id, 'Basis', 'Rouge 2', basisScore['rouge2'].precision, basisScore['rouge2'].recall, basisScore['rouge2'].fmeasure])
  # data.append([i, id, 'Basis', 'Rouge L', basisScore['rougeL'].precision, basisScore['rougeL'].recall, basisScore['rougeL'].fmeasure])
  # data.append([i, id, 'Basis', 'Rouge Lsum', basisScore['rougeLsum'].precision, basisScore['rougeLsum'].recall, basisScore['rougeLsum'].fmeasure])
  # data.append([i, id, 'TFIDF', 'Rouge 1', tfidfScore['rouge1'].precision, tfidfScore['rouge1'].recall, tfidfScore['rouge1'].fmeasure])
  # data.append([i, id, 'TFIDF', 'Rouge 2', tfidfScore['rouge2'].precision, tfidfScore['rouge2'].recall, tfidfScore['rouge2'].fmeasure])
  # data.append([i, id, 'TFIDF', 'Rouge L', tfidfScore['rougeL'].precision, tfidfScore['rougeL'].recall, tfidfScore['rougeL'].fmeasure])
  # data.append([i, id, 'TFIDF', 'Rouge Lsum', tfidfScore['rougeLsum'].precision, tfidfScore['rougeLsum'].recall, tfidfScore['rougeLsum'].fmeasure])
  # data.append([i, id, 'Div1', 'Rouge 1', diversity1Score['rouge1'].precision, diversity1Score['rouge1'].recall, diversity1Score['rouge1'].fmeasure])
  # data.append([i, id, 'Div1', 'Rouge 2', diversity1Score['rouge2'].precision, diversity1Score['rouge2'].recall, diversity1Score['rouge2'].fmeasure])
  # data.append([i, id, 'Div1', 'Rouge L', diversity1Score['rougeL'].precision, diversity1Score['rougeL'].recall, diversity1Score['rougeL'].fmeasure])
  # data.append([i, id, 'Div1', 'Rouge Lsum', diversity1Score['rougeLsum'].precision, diversity1Score['rougeLsum'].recall, diversity1Score['rougeLsum'].fmeasure])
  # data.append([i, id, 'Div2', 'Rouge 1', diversity2Score['rouge1'].precision, diversity2Score['rouge1'].recall, diversity2Score['rouge1'].fmeasure])
  # data.append([i, id, 'Div2', 'Rouge 2', diversity2Score['rouge2'].precision, diversity2Score['rouge2'].recall, diversity2Score['rouge2'].fmeasure])
  # data.append([i, id, 'Div2', 'Rouge L', diversity2Score['rougeL'].precision, diversity2Score['rougeL'].recall, diversity2Score['rougeL'].fmeasure])
  # data.append([i, id, 'Div2', 'Rouge Lsum', diversity2Score['rougeLsum'].precision, diversity2Score['rougeLsum'].recall, diversity2Score['rougeLsum'].fmeasure])



  # precisionData.append([i, id, basisScore['rouge1'].precision, basisScore['rouge2'].precision, basisScore['rougeL'].precision, basisScore['rougeLsum'].precision,
  #                              tfidfScore['rouge1'].precision, tfidfScore['rouge2'].precision, tfidfScore['rougeL'].precision, tfidfScore['rougeLsum'].precision,
  #                              diversity1Score['rouge1'].precision, diversity1Score['rouge2'].precision, diversity1Score['rougeL'].precision, diversity1Score['rougeLsum'].precision,
  #                              diversity2Score['rouge1'].precision, diversity2Score['rouge2'].precision, diversity2Score['rougeL'].precision, diversity2Score['rougeLsum'].precision])
  # recallData.append([i, id, basisScore['rouge1'].recall, basisScore['rouge2'].recall, basisScore['rougeL'].recall, basisScore['rougeLsum'].recall,
  #                           tfidfScore['rouge1'].recall, tfidfScore['rouge2'].recall, tfidfScore['rougeL'].recall, tfidfScore['rougeLsum'].recall,
  #                           diversity1Score['rouge1'].recall, diversity1Score['rouge2'].recall, diversity1Score['rougeL'].recall, diversity1Score['rougeLsum'].recall,
  #                           diversity2Score['rouge1'].recall, diversity2Score['rouge2'].recall, diversity2Score['rougeL'].recall, diversity2Score['rougeLsum'].recall])
  # fmeasureData.append([i, id, basisScore['rouge1'].fmeasure, basisScore['rouge2'].fmeasure, basisScore['rougeL'].fmeasure, basisScore['rougeLsum'].fmeasure,
  #                           tfidfScore['rouge1'].fmeasure, tfidfScore['rouge2'].fmeasure, tfidfScore['rougeL'].fmeasure, tfidfScore['rougeLsum'].fmeasure,
  #                           diversity1Score['rouge1'].fmeasure, diversity1Score['rouge2'].fmeasure, diversity1Score['rougeL'].fmeasure, diversity1Score['rougeLsum'].fmeasure,
  #                           diversity2Score['rouge1'].fmeasure, diversity2Score['rouge2'].fmeasure, diversity2Score['rougeL'].fmeasure, diversity2Score['rougeLsum'].fmeasure])



ID:  B0BJMV9BXJ
Reviews:   The convenience of simply tossing one into the laundry without measuring detergent is a game-changer. Not only do they clean my clothes thoroughly, but the Spring Meadow scent leaves them smelling fresh for days. With so many pods in one pack, it's a great value for the price. I highly recommend these Tide PODS for anyone looking for a hassle-free laundry solution! Tide PODS in Spring Meadow scent have truly transformed my laundry routine. The convenience of just tossing one pod into the washer without any measuring has been a game-changer for me. Not only do they clean my clothes thoroughly, but the fresh Spring Meadow scent lasts for days, leaving my laundry smelling wonderfully fresh. Tide Power Pods deliver powerful cleaning with every load, ensuring my clothes come out spotless and smelling great. The pre-measured pods eliminate the mess and guesswork of traditional detergent, making laundry day much simpler and more enjoyable. These are worth it. They a

In [ ]:
# dfTotal = pd.DataFrame(data, columns = ['index', 'id', 'Summary Method', 'Rouge Method', 'Precision', 'Recall', 'Fmeasure'])
# #dfTotal

# dfPrecision = pd.DataFrame(precisionData, columns = ['index', 'id', 'Basis - 1 - precision', 'Basis - 2 - precision', 'Basis - L - precision', 'Basis - Lsum - precision',
#                                                            'TF-IDF - 1 - precision', 'TF-IDF - 2 - precision', 'TF-IDF - L - precision', 'TF-IDF - Lsum - precision',
#                                                            'Div1 - 1 - precision', 'Div1 - 2 - precision', 'Div1 - L - precision', 'Div1 - Lsum - precision',
#                                                            'Div2 - 1 - precision', 'Div2 - 2 - precision', 'Div2 - L - precision', 'Div2 - Lsum - precision'])
# #dfPrecision

# dfRecall = pd.DataFrame(recallData, columns = ['index', 'id', 'Basis - 1 - recall', 'Basis - 2 - recall', 'Basis - L - recall', 'Basis - Lsum - recall',
#                                                         'TF-IDF - 1 - recall', 'TF-IDF - 2 - recall', 'TF-IDF - L - recall', 'TF-IDF - Lsum - recall',
#                                                         'Div1 - 1 - recall', 'Div1 - 2 - recall', 'Div1 - L - recall', 'Div1 - Lsum - recall',
#                                                         'Div2 - 1 - recall', 'Div2 - 2 - recall', 'Div2 - L - recall', 'Div2 - Lsum - recall'])
# #dfRecall
# dfFmeasure = pd.DataFrame(fmeasureData, columns = ['index', 'id', 'Basis - 1 - fmeasure', 'Basis - 2 - fmeasure', 'Basis - L - fmeasure', 'Basis - Lsum - fmeasure',
#                                                         'TF-IDF - 1 - fmeasure', 'TF-IDF - 2 - fmeasure', 'TF-IDF - L - fmeasure', 'TF-IDF - Lsum - fmeasure',
#                                                         'Div1 - 1 - fmeasure', 'Div1 - 2 - fmeasure', 'Div1 - L - fmeasure', 'Div1 - Lsum - fmeasure',
#                                                         'Div2 - 1 - fmeasure', 'Div2 - 2 - fmeasure', 'Div2 - L - fmeasure', 'Div2 - Lsum - fmeasure'])
# #dfFmeasure

In [ ]:
def get_or_create_spreadsheet(spreadsheet_name):
    drive_service = build('drive', 'v3', credentials=credentials)
    try:
        spreadsheet = gc.open(spreadsheet_name)
        print(f"Spreadsheet '{spreadsheet_name}' already exists.")
        # spreadsheet_id = spreadsheet.id
        # drive_service.files().delete(fileId=spreadsheet_id).execute()
        return spreadsheet
    except gspread.exceptions.SpreadsheetNotFound:
        spreadsheet = gc.create(spreadsheet_name)
        print(f"Spreadsheet '{spreadsheet_name}' created.")

        spreadsheet.add_worksheet(title="Precision", rows="100", cols="20")
        spreadsheet.add_worksheet(title="Recall", rows="100", cols="20")
        spreadsheet.add_worksheet(title="Fmeasure", rows="100", cols="20")

        # RA Notebook Folder ID
        folder_id = '1Re0yt2iK6qtK24itK6fFTo0WNbPySsSO'

        # File ID of new sheet
        file_id = spreadsheet.id

        # Update the file's parent to move it to the desired folder
        file = drive_service.files().get(fileId=file_id, fields='parents').execute()
        previous_parents = ",".join(file.get('parents'))

        # Move the file to the folder
        drive_service.files().update(fileId=file_id,
                                    addParents=folder_id,
                                    removeParents=previous_parents,
                                    fields='id, parents').execute()

        print(f"File '{spreadsheet.title}' moved to folder with ID: {folder_id}")
        return spreadsheet



In [ ]:
credentials, project = default()
gc = gspread.authorize(credentials)


spreadsheet = gc.create('My New Spreadsheet')

# Step 6: Access the worksheet and populate with data
worksheet = spreadsheet.get_worksheet(0)
df = pd.DataFrame({
    'A': [1, 2, 3],
    'B': ['x', 'y', 'z']
})
set_with_dataframe(worksheet, df)


File 'My New Spreadsheet' moved to folder with ID: 1Re0yt2iK6qtK24itK6fFTo0WNbPySsSO


In [ ]:
def export_to_sheets(dfTotal, dfPrecision, dfRecall, dfFmeasure):
  spreadsheetName = 'Rouge Scores'
  spreadsheet = get_or_create_spreadsheet(spreadsheetName)

  worksheet1 = spreadsheet.get_worksheet(0)
  set_with_dataframe(worksheet1, dfTotal)

  worksheet2 = spreadsheet.get_worksheet(1)
  set_with_dataframe(worksheet2, dfPrecision)

  worksheet3 = spreadsheet.get_worksheet(2)
  set_with_dataframe(worksheet3, dfRecall)

  worksheet4 = spreadsheet.get_worksheet(3)
  set_with_dataframe(worksheet4, dfFmeasure)

export_to_sheets(dfTotal, dfPrecision, dfRecall, dfFmeasure)

Spreadsheet 'Rouge Scores' created.
File 'Rouge Scores' moved to folder with ID: 1Re0yt2iK6qtK24itK6fFTo0WNbPySsSO


In [ ]:
#sns.violinplot(dfTotal, x='Precision', y='Score Method', inner='stick', hue='Rouge Method')


In [ ]:
#sns.violinplot(dfTotal, x='Precision', y='Rouge Method', inner='stick', hue='Rouge Method')


In [ ]:
#sns.violinplot(dfTotal, x='Precision', y='Score Method', inner='stick', hue='Score Method')
